In [1]:
from pathlib import Path

from IPython.display import display, Markdown

from five_safes_tes_workbench.workbench import Workbench
from five_safes_tes_workbench.schema.config_schema import ConfigValidationModel
from five_safes_tes_workbench.core.tes.templates.load_template import load_bunny_template, load_library_template, load_template

Imagine you had seen a task someone had run through 5S-TES that you liked.
Luckily for you, if you visit your submission layer, you can see the JSON for their task and save it.
Even more luckily, you can use the workbench to use it and modify it for your own purpose.

In [2]:
complicated_task = load_template(Path("../../../tests/test_data/multiple-executor-task.json"))

In [3]:
display(
    Markdown(
        f"""## Task
- name: {complicated_task.name}
- description: {complicated_task.description}
- Number of executors: {len(complicated_task.executors)}
        """
    )
)

## Task
- name: Test - DrugUtilisation add metrics
- description: Generate an acetaminophen cohort, add drug utilisation metrics, summarise one metric, then clean up
- Number of executors: 4
        

I imagine you don't want to send exactly the same task though.

We can change the description

In [4]:
complicated_task.description = "I'm a special snowflake"
complicated_task.description

"I'm a special snowflake"

We can look at the executors too.

In [5]:
complicated_task.executors

[ExecutorTESParams(image='ghcr.io/health-informatics-uon/omop-r-tools:edge', command=['Rscript', 'inst/scripts/generateIngredientCohortSet.R', 'du_add_metrics_test_1', '--ingredient=acetaminophen', '--gapEra=7'], workdir=None, stdin=None, stdout=None, stderr=None, env={}, ignore_error=None),
 ExecutorTESParams(image='ghcr.io/health-informatics-uon/omop-r-tools:edge', command=['Rscript', 'inst/scripts/addDrugUtilisation.R', 'du_add_metrics_test_1', '--ingredientConceptId=1125315', '--gapEra=7'], workdir=None, stdin=None, stdout=None, stderr=None, env={}, ignore_error=None),
 ExecutorTESParams(image='ghcr.io/health-informatics-uon/omop-r-tools:edge', command=['Rscript', 'inst/scripts/summariseCohort.R', 'du_add_metrics_test_1', '--variables=["days_exposed_ingredient_1125315_descendants"]', '--estimates=["mean","median","q25","q75"]', '--output-path=outputs/drug_utilisation_metrics.csv'], workdir=None, stdin=None, stdout=None, stderr=None, env={}, ignore_error=None),
 ExecutorTESParams(im

Ah, so the first task is where the ingredient is defined that the cohort is built from.

In [6]:
complicated_task.executors[0]

ExecutorTESParams(image='ghcr.io/health-informatics-uon/omop-r-tools:edge', command=['Rscript', 'inst/scripts/generateIngredientCohortSet.R', 'du_add_metrics_test_1', '--ingredient=acetaminophen', '--gapEra=7'], workdir=None, stdin=None, stdout=None, stderr=None, env={}, ignore_error=None)

I want to do the same analysis, but looking at ibuprofen instead.

In [7]:
complicated_task.executors[0].command

['Rscript',
 'inst/scripts/generateIngredientCohortSet.R',
 'du_add_metrics_test_1',
 '--ingredient=acetaminophen',
 '--gapEra=7']

In [8]:
complicated_task.executors[0].command[3] = "--ingredient=ibuprofen"
complicated_task.executors[0]

ExecutorTESParams(image='ghcr.io/health-informatics-uon/omop-r-tools:edge', command=['Rscript', 'inst/scripts/generateIngredientCohortSet.R', 'du_add_metrics_test_1', '--ingredient=ibuprofen', '--gapEra=7'], workdir=None, stdin=None, stdout=None, stderr=None, env={}, ignore_error=None)

In [9]:
wb = Workbench()

INFO | Template registered: 'hello_world'
INFO | Template registered: 'custom'
INFO | Template registered: 'simple_sql'
INFO | Template registered: 'bunny'
INFO | Template registered: 'analysis'


In [10]:
config = ConfigValidationModel(
    project="pretend_project",
    tes_base_url="http://localhost:8085/",
    tres=["TRE 1", "TRE 2"]
)

In [11]:
wb._task_builder.build(config=config, params=complicated_task)

INFO | TES Task built successfully
INFO | TES payload:
{
   "name": "Test - DrugUtilisation add metrics",
   "description": "I'm a special snowflake",
   "outputs": [
      {
         "url": "s3://",
         "path": "/outputs",
         "type": "DIRECTORY",
         "name": "Query Results",
         "description": "Drug utilisation metric summary"
      }
   ],
   "executors": [
      {
         "image": "ghcr.io/health-informatics-uon/omop-r-tools:edge",
         "command": [
            "Rscript",
            "inst/scripts/generateIngredientCohortSet.R",
            "du_add_metrics_test_1",
            "--ingredient=ibuprofen",
            "--gapEra=7"
         ],
         "env": {}
      },
      {
         "image": "ghcr.io/health-informatics-uon/omop-r-tools:edge",
         "command": [
            "Rscript",
            "inst/scripts/addDrugUtilisation.R",
            "du_add_metrics_test_1",
            "--ingredientConceptId=1125315",
            "--gapEra=7"
         ],
     